# 3D Image Registration in Brain Tumor Segmentation

3D image registration is essential in medical imaging, including in brain tumor segmentation. This technique aligns multiple 3D images from different scans or modalities to create a unified, multi-dimensional view of a patient's anatomy. Key modalities include T1-weighted (T1), T1-weighted with gadolinium contrast (T1Gd), T2-weighted (T2), and Fluid Attenuated Inversion Recovery (FLAIR).

### Background of Modalities

- **T1 and T1Gd**: 
  - T1 images provide detailed anatomy of the brain, showing structures of white and gray matter.
  - T1Gd images use a contrast agent, gadolinium, which enhances the visibility of tumors due to the disrupted blood-brain barrier in tumor areas.

- **T2**: 
  - These images are fluid-sensitive, ideal for visualizing edema and infiltrated areas not clearly visible on T1 images.

- **FLAIR**: 
  - FLAIR images are excellent for differentiating between cerebrospinal fluid and pathological changes, particularly useful for detecting lesions near ventricles and subarachnoid spaces.

### Importance of 3D Image Registration

 **Comprehensive Tumor Analysis**:
   - Registering these modalities provides a complete picture of the tumor, combining the distinct advantages of each modality for accurate tumor delineation and volume estimation.

 **Enhanced Monitoring and Research**:
   - Effective registration allows for consistent monitoring of tumor progression or regression and supports the development of automated segmentation algorithms by providing aligned datasets for training.

In summary, 3D image registration is crucial for accurately detecting, delineating, and treating brain tumors in clinical and research settings, leveraging the unique strengths of modalities like T1, T1Gd, T2, and FLAIR.


## BraTS like registration toolkit built upon HD-BET and ANTsPY

- BraTS elaborated as Brain Tumor Segmentation Challenge
- HD-BET is deep learning based brain extraction tool (https://github.com/MIC-DKFZ/HD-BET)
- ANTsPY: Python based medical imaging registration library
- Re-orientation to LPS/RAI
- Image registration to SRI-24 Atlas  which includes the following steps
- N4 Bias correction (This is a TEMPORARY STEP, and is not applied in the final co-registered output images. It is only use to facilitate optimal registration.)
- Rigid Registration of T1, T2, FLAIR to T1CE
- Rigid Registration of T1CE to SRI-24 atlas 
- Applying transformation to the reoriented images

Inspiration from https://github.com/neuronflow/BraTS-Toolkit

#### Loading necessary modules

In [1]:
import os
os.environ['MKL_THREADING_LAYER'] = 'GNU'

In [2]:
import shutil
import tempfile
import sys
import math
import pandas as pd
import subprocess
import pdb
import nibabel as nib
import ants
import copy
from subprocess import Popen
from subprocess import PIPE
import multiprocessing as mp
from datetime import datetime
from uuid import uuid4

import numpy as np
import glob
import monai
import scipy
from scipy import ndimage
import SimpleITK as sitk


from collections import OrderedDict
from typing import Callable, Sequence, Type, Union

## Function Overview: `brats_ants_mni152betafter_registration`

The Python function `brats_ants_mni152betafter_registration` processes MRI brain images for a given subject by performing image registration and bias field correction using the ANTsPy library.

### Workflow Breakdown

1. **Reading and Initializing Images**:
   - Reads the atlas brain image and its mask from specified paths, already oriented in 'RAI' format.
   - Retrieves the subject ID from the provided dictionary `aSubjectDct` and creates a directory for the subject in the specified `save_dir` if it doesn't exist.

2. **Bias Field Correction**:
   - Reads T1-weighted contrast-enhanced (T1ce), T1-weighted (T1), T2-weighted (T2), and FLAIR images from paths specified in `aSubjectDct`.
   - Each image undergoes bias field correction using ANTsPy's `n4_bias_field_correction` function.

3. **Temporary Output Directory**:
   - Creates a temporary directory to store intermediate results, including subdirectories for different registration outputs.

4. **Image Registration**:
   - Performs image registration using ANTsPy's `registration` function. The transformations include:
     - Registering T1, T2, and FLAIR images to the T1ce image using a 'DenseRigid' transformation.
     - Registering the T1ce image to the atlas image using a 'DenseRigid' transformation.

5. **Applying Transformations and Saving Results**:
   - Applies the computed transformations to the respective images (T1, T2, and FLAIR images) to align them with the atlas image.
   - Saves the registered images to the subject's directory. For the T1ce image, additional processing includes applying a mask, filling holes, and saving a version of masked image of the corresponding object.
   - Runs an external command (`hd-bet`) to perform brain extraction on the registered T1ce image, reads the resulting mask, and applies to the corresponding T1, T2, and FLAIR images.

### Purpose

This function is tailored for processing brain MRI images in a research setting, particularly for tasks involving image registration and normalization to a standard atlas. It automates the workflow of preparing images for further analysis, such as segmentation or other forms of image-based studies.

In [3]:
def brats_ants_mni152betafter_registration(aSubjectDct, AtlasPath:str, AtlasMaskPath:str, save_dir:str):

    
    atlas_brain_sri24_bet = ants.image_read(AtlasPath) ## already in reorient = 'RAI'
    atlas_brain_sri24_betMask =  ants.image_read(AtlasMaskPath) ##already in reorient = 'RAI'

    
    aSubId = aSubjectDct['SubjectID']
    print(f"Start processing {aSubId}\n\n")
    aSubDir = os.path.join(save_dir, aSubId)
    if not os.path.exists(aSubDir):
        os.makedirs(aSubDir)
    
    
    
    neighb_struct = np.ones((3,3,3))

    #print(aSubjectDct)
    
    
    '''Running registration process'''
    
    T1ceImg = ants.image_read(aSubjectDct['t1cwPath'], reorient="RAI")
    T1ceImg_n4 = ants.n4_bias_field_correction(T1ceImg)
   
    
    T1Img = ants.image_read(aSubjectDct['t1wPath'], reorient="RAI")
    T1Img_n4 = ants.n4_bias_field_correction(T1Img)
    
    
    T2Img= ants.image_read(aSubjectDct['t2wPath'], reorient="RAI")
    T2Img_n4 = ants.n4_bias_field_correction(T2Img)
   
    
    FlairImg = ants.image_read(aSubjectDct['flairPath'], reorient="RAI")
    FlairImg_n4 = ants.n4_bias_field_correction(FlairImg)
   
    
        
    '''Setting directory where temporary files will be saved'''
    
    SubOutDir = datetime.now().strftime('%Y%m-%d%H-%M%S-') + str(uuid4())
    outputDirectory = os.path.join(aSubDir,f"TempsOutputWithMaskANTsPy_BraTS21_{SubOutDir}")
    if not os.path.exists(outputDirectory):
        os.mkdir(outputDirectory)
        os.mkdir(f"{outputDirectory}/t1_2t1Gd")
        os.mkdir(f"{outputDirectory}/t2_2t1Gd")
        os.mkdir(f"{outputDirectory}/flair_2t1Gd")
        os.mkdir(f"{outputDirectory}/t1c_2sri24")
                                   
    
    
    '''T1, T2, and flair to t1ce'''
    
    
    mytx_t1_2t1Gd = ants.registration(fixed=T1ceImg_n4, moving=T1Img_n4,  type_of_transform='DenseRigid',\
                         verbose = True, outprefix = f"{outputDirectory}/t1_2t1Gd/t1_2t1Gd") ##'SyN' 'TRSAA', 'SyNRA','Rigid'
    print(mytx_t1_2t1Gd)
    
    mytx_t2_2t1Gd = ants.registration(fixed=T1ceImg_n4, moving=T2Img_n4, type_of_transform='DenseRigid',\
                     verbose = True, outprefix = f"{outputDirectory}/t2_2t1Gd/t2_2t1Gd") ##'SyN' 'TRSAA', 'SyNRA','Rigid'
    
    mytx_flair_2t1Gd = ants.registration(fixed=T1ceImg_n4, moving=FlairImg_n4, type_of_transform='DenseRigid',\
                 verbose = True, outprefix = f"{outputDirectory}/flair_2t1Gd/flair_2t1Gd") ##'SyN' 'TRSAA', 'SyNRA','Rigid'
    
    
    
    '''Registration'''
    
    mytx_t1Gd_2sri24 = ants.registration(fixed=atlas_brain_sri24_bet, moving=T1ceImg_n4, type_of_transform='DenseRigid',\
                             verbose = True, outprefix = f"{outputDirectory}/t1c_2sri24/t1c_2sri24") ##'SyN' 'TRSAA', 'SyNRA', 'Rigid'
    
    
    
    '''Applying registration and saving'''

    '''T1Gd to sri24'''
    #masked once only
    #T1ceImg = T1ceImg*T1ceMaskImg
    applyRegdT1Gd = ants.apply_transforms(fixed=atlas_brain_sri24_bet, moving=T1ceImg, transformlist=mytx_t1Gd_2sri24['fwdtransforms'], interpolator='linear')
    #applyRegdT1Gd*atlas_brain_sri24_betMask
    ants.image_write(applyRegdT1Gd, f'{aSubDir}/{aSubId}_t1GdRegdWithSkull.nii.gz', ri = True)
    t1GdFinalMask = applyRegdT1Gd.clone()*atlas_brain_sri24_betMask.clone()
    
    
    t1GdFinalMask = t1GdFinalMask.new_image_like(ndimage.binary_fill_holes(np.where(t1GdFinalMask.numpy()>0, 1., 0.), structure = neighb_struct).astype(np.float32))
    ants.image_write(t1GdFinalMask, f'{aSubDir}/{aSubId}_t1GdBrainROI.nii.gz', ri = True)

    ## using GPU 
    #proc_status_t1Gdbet = subprocess.run(['hd-bet', '-i', f'{aSubDir}/{aSubId}_t1GdRegdWithSkull.nii.gz', '-o', f'{aSubDir}/{aSubId}_t1Gd.nii.gz', '-s', '1'], capture_output=True, text=True)
    ## Using CPU
    subprocess.run(['hd-bet', '-i', f'{aSubDir}/{aSubId}_t1GdRegdWithSkull.nii.gz', '-o', f'{aSubDir}/{aSubId}_t1Gd.nii.gz', '-device', 'cpu', '--disable_tta', '--save_bet_mask'])   
    proc_status_t1Gdbet = subprocess.run(['mv', f'{aSubDir}/{aSubId}_t1Gd_bet.nii.gz', f'{aSubDir}/{aSubId}_t1Gd_mask.nii.gz'])
    hbet_T1ceMaskImg = ants.image_read(f'{aSubDir}/{aSubId}_t1Gd_mask.nii.gz')
    
    
    '''T1, T2, flair to T1GD and then t1gd-sri24'''
    
    '''for t1'''
    applyRegdT1 = ants.apply_transforms(fixed=T1ceImg_n4, moving=T1Img, transformlist=mytx_t1_2t1Gd['fwdtransforms'], interpolator='linear')
    #applyRegdT1 = applyRegdT1*hbet_T1ceMaskImg
    applyRegdT1_sri24 = ants.apply_transforms(fixed=atlas_brain_sri24_bet, moving=applyRegdT1, transformlist=mytx_t1Gd_2sri24['fwdtransforms'], interpolator='linear')
    ants.image_write(applyRegdT1_sri24*hbet_T1ceMaskImg, f'{aSubDir}/{aSubId}_t1.nii.gz', ri = True)
    

    
    
    '''for t2'''
    applyRegdT2 = ants.apply_transforms(fixed=T1ceImg_n4, moving=T2Img, transformlist=mytx_t2_2t1Gd['fwdtransforms'], interpolator='linear')
    #applyRegdT2 = applyRegdT2*hbet_T1ceMaskImg
    applyRegdT2_sri24 = ants.apply_transforms(fixed=atlas_brain_sri24_bet, moving=applyRegdT2, transformlist=mytx_t1Gd_2sri24['fwdtransforms'], interpolator='linear')
    ants.image_write(applyRegdT2_sri24*hbet_T1ceMaskImg, f'{aSubDir}/{aSubId}_t2.nii.gz', ri = True)

    
    
    '''for flair'''

    applyRegdFlair = ants.apply_transforms(fixed=T1ceImg_n4, moving=FlairImg, transformlist=mytx_flair_2t1Gd['fwdtransforms'], interpolator='linear')
    #applyRegdFlair = applyRegdFlair*hbet_T1ceMaskImg
    applyRegdFlair_sri24 = ants.apply_transforms(fixed=atlas_brain_sri24_bet, moving=applyRegdFlair, transformlist=mytx_t1Gd_2sri24['fwdtransforms'], interpolator='linear')
    ants.image_write(applyRegdFlair_sri24*hbet_T1ceMaskImg, f'{aSubDir}/{aSubId}_flair.nii.gz', ri = True)

    
    #shutil.rmtree(outputDirectory)
    print('#'*80)

In [4]:
#AtlasPath = '/home/mmiv-ml/saruarlive/IDHRadiogenomics2022/assets/T1_sri24_BraTS_bet.nii.gz'
AtlasPath = '../data/sri24_spm8_selected/T1_sri24_BraTS.nii.gz'
AtlasMaskPath =  '../data/sri24_spm8_selected/T1_sri24_BraTS_bet_mask.nii.gz'

save_dir = '../results/BraTSLikeProcess_HDBETAfterReg'


aDCT = {'SubjectID':"EGD-0117", 't1wPath': "../data/EGD-0117/T1.nii.gz",\
        't1cwPath': "../data/EGD-0117/T1GD.nii.gz",\
        't2wPath': "../data/EGD-0117/T2.nii.gz",\
        'flairPath': "../data/EGD-0117/FLAIR.nii.gz"}

brats_ants_mni152betafter_registration(aDCT, AtlasPath, AtlasMaskPath, save_dir)
print('##'*20)

Start processing EGD-0117


antsRegistration -d 3 -r [0x7f75fd710588,0x7f75fd710ba8,1] -m mattes[0x7f75fd710588,0x7f75fd710ba8,1,32,regular,1.0] -t Rigid[0.25] -c 2100x1200x1200x10 -s 3x2x1x0 -f 6x4x2x1 -u 1 -z 1 -o [../results/BraTSLikeProcess_HDBETAfterReg/EGD-0117/TempsOutputWithMaskANTsPy_BraTS21_202501-2907-4002-2611ce23-ef0d-48d1-83f8-79f11215a866/t1_2t1Gd/t1_2t1Gd,0x7f75fd710b88,0x7f75fd710608] -x [NA,NA] --float 1 --write-composite-transform 0 -v 1
All_Command_lines_OK
Using single precision for computations.
The composite transform comprises the following transforms (in order): 
  1. Center of mass alignment using fixed image: 0x7f75fd710588 and moving image: 0x7f75fd710ba8 (type = Euler3DTransform)
  Reading mask(s).
    Registration stage 0
      No fixed mask
      No moving mask
  number of levels = 4
  fixed image: 0x7f75fd710588
  moving image: 0x7f75fd710ba8
Dimension = 3
Number of stages = 1
Use histogram matching = true
Winsorize image intensities = false
  Lower quan

/home/har010/miniconda3/envs/3d-image-registration-segmentation/lib/python3.11/site-packages/nnunetv2/inference/predict_from_raw_data.py:84: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experiment


########################
If you are using hd-bet, please cite the following papers:

Isensee F, Schell M, Tursunova I, Brugnara G, Bonekamp D, Neuberger U, Wick A, Schlemmer HP, Heiland S, Wick W, Bendszus M, Maier-Hein KH, Kickingereder P. Automated brain extraction of multi-sequence MRI using artificial neural networks. arXiv preprint arXiv:1901.11341, 2019.

Isensee, F., Jaeger, P. F., Kohl, S. A., Petersen, J., & Maier-Hein, K. H. (2021). nnU-Net: a self-configuring method for deep learning-based biomedical image segmentation. Nature methods, 18(2), 203-211.
########################

perform_everything_on_device=True is only supported for cuda devices! Setting this to False
There are 1 cases in the source folder
I am process 0 out of 1 (max process ID is 0, we start counting with 0!)
There are 1 cases that I would like to predict

Predicting EGD-0117_t1Gd_bet.nii.gz:
perform_everything_on_device: False


100%|██████████| 6/6 [00:46<00:00,  7.69s/it]


sending off prediction to background worker for resampling and export
done with EGD-0117_t1Gd_bet.nii.gz
################################################################################
########################################
